# Dia 19 — Wayback Machine e funções de data e hora

**🃏 Cartas:** 19 · 40 · 61 &nbsp;·&nbsp; **📘 Original:** *Day 19. Wayback Machine and time/date functions*
&nbsp;·&nbsp; [repositório](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias) &nbsp;·&nbsp; [curso original do cipher387](https://github.com/cipher387/python-for-OSINT-21-days)

---

## 📖 Capítulo 19 — O site que sumiu

— Clica no link da loja — pede a Iara.

Você clica. **404.** A MegaOferta Brasil não existe mais.

— Era isso que ele queria — ela diz, sem surpresa. — Site falso tem prazo de validade. Some antes da
primeira intimação. Só que a internet tem memória.

Ela abre o site do Internet Archive e digita o domínio. Aparece uma régua do tempo com bolinhas
azuis: cinco capturas, entre outubro e dezembro.

— Cada bolinha é uma foto do site num momento. Mas olhar uma por uma no navegador não é análise. A
API devolve a lista inteira em JSON: data, status, tamanho da página. E o **tamanho** fala. Uma
vitrine cheia de produtos pesa seis mil bytes; um aviso de "em manutenção" pesa oitocentos.

Ela pega o calendário de mesa e circula uma data em novembro.

— Agora o mais importante: **datas**. Junta o nascimento do domínio, as capturas e as denúncias numa
linha só. E me diz uma coisa: em que dia caiu a Black Friday do ano passado? Aposto que as
denúncias se amontoam em volta dela.

In [ ]:
#@title ⚙️ Preparar o ambiente (rode esta célula primeiro)
# No Colab: baixa o repositório e entra na pasta do dia.
# No Jupyter local: o notebook já está na pasta do dia, então não faz nada.
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias"
if "google.colab" in sys.modules:
    destino = "/content/python-para-osint-21-dias"
    if not os.path.isdir(destino):
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    os.chdir(f"{destino}/dia-19-wayback-e-datas")

PASTA_NB = str(Path.cwd())          # usada pelas células para voltar sempre à pasta certa
print("📂 Pasta do dia:", PASTA_NB)
print("🐍 Python", sys.version.split()[0])

## 🎯 Objetivos

- Baixar cópias antigas de um site com o pacote `wayback`, como no curso original.
- Consultar a **API CDX** da Wayback Machine só com a biblioteca padrão.
- Converter timestamps `AAAAMMDDhhmmss` e Unix em datas com fuso.
- Trabalhar com `America/Sao_Paulo`, `dd/mm/aaaa` e dias da semana em português sem `locale`.
- Montar a linha do tempo do caso e calcular a Black Friday com `calendar`.

## 🧠 Conceito em 2 minutos

A **Wayback Machine** (Internet Archive) guarda "fotos" de páginas web ao longo do tempo. Cada foto
é um **memento**. Para saber quais existem, pergunte ao índice **CDX**:

```
https://web.archive.org/cdx/search/cdx?url=gov.br&output=json&limit=5
```

A resposta é uma lista de listas, e a **primeira linha é o cabeçalho**:
`urlkey, timestamp, original, mimetype, statuscode, digest, length`.
O `timestamp` (`20251128233001`) está sempre em **UTC**. O `digest` é a impressão digital do
conteúdo: muda quando a página muda. Para abrir a cópia: `https://web.archive.org/web/<timestamp>/<url>`.

**Datas em Python** têm uma pegadinha: um `datetime` pode ser **ingênuo** (sem fuso, "hora do
relógio de alguém") ou **consciente** (com fuso). Evidência sempre com fuso:

| Precisa de | Use |
|---|---|
| Fuso de Brasília | `ZoneInfo("America/Sao_Paulo")` (no Windows, `pip install tzdata` ou -03:00 fixo) |
| Texto → data | `datetime.strptime("28/11/2025", "%d/%m/%Y")` |
| Data → texto | `f"{d:%d/%m/%Y}"` ou `d.strftime(...)` |
| Somar dias | `d + timedelta(days=30)` |
| Unix → data | `datetime.fromtimestamp(1764375240, FUSO)` |

O Brasil **não tem mais horário de verão desde 2019**, mas datas antigas ainda estão em -02:00 nos
meses de verão. O `ZoneInfo` sabe disso; o fuso fixo, não.

## 🥋 Nível 1 — Exercício original (cipher387)

| Arquivo | O que faz |
|---|---|
| [`original/date_time.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-19-wayback-e-datas/original/date_time.py) | Data e hora atuais, ano, mês e data por extenso com `strftime` |
| [`original/download_mementos.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-19-wayback-e-datas/original/download_mementos.py) | Baixa para arquivos `.html` as cópias de `nasa.gov` feitas antes de 1999 (pacote `wayback`) |

```bash
pip install wayback
cd dia-19-wayback-e-datas/original
python date_time.py
python download_mementos.py     # demora; grava vários .html na pasta atual
```

> 💡 No `date_time.py`, `%A %d %B %Y` sai em inglês ("Monday 12 October 2026"). O `br/01` mostra como
> ter "segunda-feira, 12 de outubro de 2026" em qualquer computador.

> ⚠️ O `download_mementos.py` usa `open(fileName, "a")`: rodar duas vezes **acrescenta** o HTML de
> novo no mesmo arquivo. E, no Windows, sem `encoding="utf-8"` pode dar erro de acento.

In [ ]:
#@title 📦 Instalar as bibliotecas do exercício original
%pip install -q wayback

### `original/date_time.py` — Data e hora atuais com datetime

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Data e hora atuais com datetime
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_19/date_time.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# ──────────────────────────────────────────────────────────────────────────
# 💬 datetime vem com o Python; não precisa instalar.
import datetime

# 💬 Hora local do computador, SEM fuso horário anexado ('ingênua').
# 💬 No Colab, o relógio está em UTC: 3 horas à frente de Brasília.
currentTime = datetime.datetime.now()

print(currentTime)

# 💬 year, month, day, hour... são atributos numéricos; str() converte para texto.
print("Current Year: "+str(currentTime.year))

print("Current Month: "+str(currentTime.month))

# 💬 strftime formata a data. %A e %B saem em INGLÊS (Monday, October)
# 💬 a menos que você mude o locale. Veja br/01_datas_br.py para pt-BR sem locale.
print(currentTime.strftime("%A %d %B %Y"))

### `original/download_mementos.py` — Baixar cópias antigas de um site da Wayback Machine

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Baixar cópias antigas de um site da Wayback Machine
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_19/download_mementos.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# Requer: pip install wayback  (demora: baixa várias páginas da Wayback Machine)
# ──────────────────────────────────────────────────────────────────────────
# 💬 Pacote 'wayback' (mantido pela EDGI) que conversa com a API do Internet Archive.
import wayback
from datetime import datetime

# 💬 O cliente cuida das requisições e dos limites de velocidade.
client = wayback.WaybackClient()

# 💬 search() consulta o índice CDX: cada 'record' é UMA captura (data, URL, status...).
# 💬 to_date limita às capturas feitas antes de 01/01/1999.
for record in client.search('http://nasa.gov', to_date=datetime(1999, 1, 1)):

    # 💬 'Memento' é a cópia arquivada em si: aqui o HTML é baixado de verdade.
    memento = client.get_memento(record)

    # 💬 Troca '/' por '-' porque '/' não pode aparecer em nome de arquivo.
    fileName=memento.memento_url.replace("/","-")+".html"

    # 💬 Modo 'a' (acrescentar). Sem encoding='utf-8', no Windows pode dar erro de acento.
    memento_file = open(fileName, "a")

    memento_file.write(memento.text)

    memento_file.close()

    print (fileName)

## 🇧🇷 Nível 2 — Camada Brasil

Só **biblioteca padrão**. O script 02 consulta a API ao vivo e tem amostra offline; o 03 é 100%
offline (dados do caso).

| Arquivo | O que faz |
|---|---|
| [`br/01_datas_br.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-19-wayback-e-datas/br/01_datas_br.py) | Fuso America/Sao_Paulo com plano B, strptime/strftime, timestamp Unix, timedelta, dias da semana em português e o horário de verão de 2018 |
| [`br/02_wayback_cdx.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-19-wayback-e-datas/br/02_wayback_cdx.py) | API CDX para uma página `.gov.br`: cabeçalho, conversão de timestamp, capturas por ano, status, digests e links das cópias |
| [`br/03_linha_do_tempo_caso.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-19-wayback-e-datas/br/03_linha_do_tempo_caso.py) | **O caso:** RDAP + Wayback + denúncias numa linha do tempo; queda do `length`; 404 final; Black Friday com `calendar` |

```bash
cd dia-19-wayback-e-datas
python br/02_wayback_cdx.py
python br/03_linha_do_tempo_caso.py
```

**O que você deve ver no caso:** domínio registrado em 02/10/2025; primeira vítima 10 dias depois;
Black Friday em **28/11/2025**; **12 das 37 denúncias (32%)** entre 25/11 e 01/12, mais que o dobro
da média diária; em 01/12 a página encolhe 86% ("manutenção"); em 05/12, 404.

> 💡 Os campos com formulário à direita (`#@param`) podem ser editados sem mexer no código. Para testar sem internet, rode antes: `os.environ['OSINT_OFFLINE'] = '1'`.

### `br/01_datas_br.py` — Datas e horas do jeito brasileiro: fuso, dd/mm/aaaa, timestamp Unix e dias da semana em português.

In [ ]:
%cd -q {PASTA_NB}
"""
Datas e horas do jeito brasileiro: fuso, dd/mm/aaaa, timestamp Unix e dias da semana em português.

Carta 40 (rank A) — Jutsu da Linha do Tempo

O que você aprende:
  • a diferença entre data "ingênua" (sem fuso) e "consciente" (com fuso);
  • America/Sao_Paulo com zoneinfo (e o plano B para Windows sem tzdata);
  • strptime (texto → data) e strftime (data → texto) com dd/mm/aaaa;
  • timestamp Unix, timedelta e dias da semana em português SEM depender do locale;
  • por que o horário de verão importa em datas antigas (ele foi extinto em 2019).

Como rodar (dentro da pasta dia-19-wayback-e-datas):
    python br/01_datas_br.py
"""

from datetime import date, datetime, timedelta, timezone

# ── Parâmetros ────────────────────────────────────────────────────────────
DATA_TEXTO = "28/11/2025 21:14"  #@param {type:"string"}
DIAS_PARA_SOMAR = 30  #@param {type:"integer"}

# ── 1. O fuso de Brasília ─────────────────────────────────────────────────
# zoneinfo (Python 3.9+) usa a base oficial de fusos (IANA). No Windows essa base
# pode não existir: aí "pip install tzdata" resolve, ou usamos -03:00 fixo.
try:
    from zoneinfo import ZoneInfo
    FUSO = ZoneInfo("America/Sao_Paulo")
    FUSO_COMPLETO = True
except Exception:
    FUSO = timezone(timedelta(hours=-3), "BRT")
    FUSO_COMPLETO = False
    print("⚠️ Base de fusos ausente: usando UTC-3 fixo (datas antes de 2019 podem errar 1 hora).")

# Dias e meses em português, na mão: strftime("%A") depende do "locale" do sistema,
# que no Colab e no GitHub Actions é inglês. Lista própria = mesmo resultado em qualquer lugar.
DIAS_SEMANA = ["segunda-feira", "terça-feira", "quarta-feira", "quinta-feira",
               "sexta-feira", "sábado", "domingo"]           # weekday(): 0 = segunda
MESES = ["janeiro", "fevereiro", "março", "abril", "maio", "junho", "julho",
         "agosto", "setembro", "outubro", "novembro", "dezembro"]


def por_extenso(d):
    """date/datetime → 'sexta-feira, 28 de novembro de 2025'."""
    return f"{DIAS_SEMANA[d.weekday()]}, {d.day} de {MESES[d.month - 1]} de {d.year}"


# ── 2. Agora: ingênuo × consciente ────────────────────────────────────────
ingenua = datetime.now()                 # sem fuso: "hora do computador", seja ela qual for
consciente = datetime.now(FUSO)          # com fuso: sabe que é -03:00
utc = datetime.now(timezone.utc)
print("🕐 Agora")
print(f"   ingênua (sem fuso):     {ingenua:%d/%m/%Y %H:%M:%S}   tzinfo={ingenua.tzinfo}")
print(f"   Brasília (consciente):  {consciente:%d/%m/%Y %H:%M:%S} {consciente:%z}")
print(f"   UTC:                    {utc:%d/%m/%Y %H:%M:%S} {utc:%z}")
print(f"   📅 {por_extenso(consciente)}")
print("   💡 No Colab a 'ingênua' sai em UTC (3 h à frente). Em evidência, SEMPRE grave com fuso.")

# ── 3. Texto → data (strptime) e data → texto (strftime) ──────────────────
# %d dia · %m mês · %Y ano com 4 dígitos · %H hora 0–23 · %M minuto · %S segundo
momento = datetime.strptime(DATA_TEXTO, "%d/%m/%Y %H:%M").replace(tzinfo=FUSO)
print(f"\n🔤 '{DATA_TEXTO}' → {momento.isoformat()}")
print(f"   {por_extenso(momento)}, às {momento:%Hh%M}")
print(f"   Formato ISO 8601 (o melhor para ordenar e trocar dados): {momento:%Y-%m-%d}")

# ── 4. Timestamp Unix ─────────────────────────────────────────────────────
# Segundos desde 01/01/1970 00:00 UTC. Aparece em logs, metadados de foto, APIs, Discord...
ts = int(momento.timestamp())
print(f"\n⏱️ Timestamp Unix: {ts}")
print(f"   de volta:        {datetime.fromtimestamp(ts, FUSO):%d/%m/%Y %H:%M} (Brasília)")
print(f"   o mesmo instante {datetime.fromtimestamp(ts, timezone.utc):%d/%m/%Y %H:%M} (UTC)")
# Timestamps em MILISSEGUNDOS (13 dígitos) são comuns em JavaScript: divida por 1000.
print(f"   em milissegundos: {ts * 1000}  → datetime.fromtimestamp(ms / 1000, FUSO)")

# ── 5. Contas com datas (timedelta) ───────────────────────────────────────
depois = momento + timedelta(days=DIAS_PARA_SOMAR)
print(f"\n➕ {DIAS_PARA_SOMAR} dias depois: {por_extenso(depois)}")
inicio_caso = date(2025, 10, 2)          # criação do domínio (Dia 15)
print(f"➖ Do registro do domínio até {momento:%d/%m}: {(momento.date() - inicio_caso).days} dias")

# ── 6. Horário de verão: por que o fuso "de verdade" importa ──────────────
# O Brasil teve horário de verão até fevereiro de 2019 (extinto pelo Decreto nº 9.772/2019).
# Em dezembro de 2018 Brasília estava em -02:00, não -03:00.
antigo = datetime(2018, 12, 15, 12, 0, tzinfo=FUSO)
print(f"\n☀️ 15/12/2018 12:00 em Brasília tinha deslocamento {antigo:%z}"
      f"{'  ← horário de verão!' if FUSO_COMPLETO else '  (fuso fixo: não sabe do verão)'}")
print(f"   15/12/2025 12:00 em Brasília tem deslocamento {datetime(2025, 12, 15, 12, tzinfo=FUSO):%z}")
print("   💡 Ao analisar evidência antiga com fuso fixo -03:00, você pode errar a hora em 1 h.")

### `br/02_wayback_cdx.py` — Wayback Machine sem instalar nada: a API CDX do Internet Archive.

In [ ]:
%cd -q {PASTA_NB}
"""
Wayback Machine sem instalar nada: a API CDX do Internet Archive.

Carta 40 (rank A) — Jutsu da Linha do Tempo (parte 2)

O que você aprende:
  • a perguntar à API CDX "quais capturas existem desta página?";
  • a ler a resposta JSON, em que a PRIMEIRA linha é o cabeçalho;
  • a converter o timestamp AAAAMMDDhhmmss em data de verdade;
  • a contar capturas por ano e a montar o link para abrir cada cópia arquivada;
  • a perceber mudanças de conteúdo pelo "digest" e pelo "length".

Como rodar (dentro da pasta dia-19-wayback-e-datas):
    python br/02_wayback_cdx.py
    OSINT_OFFLINE=1 python br/02_wayback_cdx.py
"""

import json
import os
import urllib.parse
import urllib.request
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

# ── Parâmetros ────────────────────────────────────────────────────────────
URL_ALVO = "gov.br"  #@param {type:"string"}
LIMITE = 200  #@param {type:"integer"}
APENAS_STATUS_200 = False  #@param {type:"boolean"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
OFFLINE = os.environ.get("OSINT_OFFLINE") == "1"


def buscar_json(url, amostra):
    """Baixa um JSON da internet; se falhar (ou OFFLINE=1), lê a amostra local."""
    if not OFFLINE:
        try:
            pedido = urllib.request.Request(url, headers={"User-Agent": "python-para-osint-21-dias/estudo"})
            with urllib.request.urlopen(pedido, timeout=30) as resposta:
                texto = resposta.read().decode("utf-8")
                # Sem nenhuma captura, a API devolve corpo vazio (e não uma lista vazia).
                return json.loads(texto) if texto.strip() else []
        except Exception as erro:
            print(f"⚠️ MODO OFFLINE: a consulta falhou ({erro}). Usando a amostra local.")
    else:
        print("⚠️ MODO OFFLINE: usando a amostra local.")
    print("   (amostra ILUSTRATIVA no formato real da API CDX: datas e tamanhos não são os verdadeiros)")
    return json.loads((PASTA_DIA / "dados" / amostra).read_text(encoding="utf-8"))


# ── 1. Montar a consulta ──────────────────────────────────────────────────
# A API CDX aceita parâmetros na URL. Os principais:
#   url=      página (ou domínio) consultado
#   output=json   resposta em JSON (o padrão é texto separado por espaços)
#   limit=    quantas linhas no máximo
#   filter=statuscode:200   só capturas que deram certo
#   collapse=digest         esconde capturas idênticas seguidas (mesmo conteúdo)
parametros = {"url": URL_ALVO, "output": "json", "limit": LIMITE}
if APENAS_STATUS_200:
    parametros["filter"] = "statuscode:200"
url = "https://web.archive.org/cdx/search/cdx?" + urllib.parse.urlencode(parametros)
print(f"🔎 {url}\n")

linhas = buscar_json(url, "wayback_cdx_gov_br.json")
if len(linhas) < 2:
    raise SystemExit("Nenhuma captura encontrada para esse endereço.")

# ── 2. Primeira linha = cabeçalho ─────────────────────────────────────────
cabecalho, dados = linhas[0], linhas[1:]
print(f"🧾 Colunas: {cabecalho}")
# zip(cabecalho, linha) casa cada nome de coluna com o valor: vira um dicionário.
capturas = [dict(zip(cabecalho, linha)) for linha in dados]
print(f"📸 {len(capturas)} capturas recebidas.\n")


def quando(timestamp):
    """'20251128233001' → datetime em UTC (a Wayback registra tudo em UTC)."""
    return datetime.strptime(timestamp, "%Y%m%d%H%M%S").replace(tzinfo=timezone.utc)


# ── 3. Primeira, última e contagem por ano ────────────────────────────────
primeira, ultima = capturas[0], capturas[-1]
print(f"🥇 Primeira captura: {quando(primeira['timestamp']):%d/%m/%Y %H:%M} UTC  (status {primeira['statuscode']})")
print(f"🕒 Última captura:   {quando(ultima['timestamp']):%d/%m/%Y %H:%M} UTC  (status {ultima['statuscode']})")

por_ano = Counter(c["timestamp"][:4] for c in capturas)       # os 4 primeiros dígitos = ano
print("\n📊 Capturas por ano:")
maior = max(por_ano.values())
for ano in sorted(por_ano):
    print(f"   {ano} {'▇' * round(por_ano[ano] / maior * 30):<30} {por_ano[ano]}")

# ── 4. Status e mudanças de conteúdo ──────────────────────────────────────
print(f"\n🚦 Status HTTP: {dict(Counter(c['statuscode'] for c in capturas))}")
# digest = impressão digital (SHA-1, em base32) do conteúdo. Digest diferente = página mudou.
versoes = len({c["digest"] for c in capturas})
print(f"🧬 Versões diferentes de conteúdo (digests distintos): {versoes}")

# ── 5. Links para abrir as cópias ─────────────────────────────────────────
# https://web.archive.org/web/<timestamp>/<url original>  → a página como estava naquele dia.
# Acrescentar "id_" ao timestamp traz o HTML ORIGINAL, sem a barra de navegação da Wayback.
print("\n🔗 Abra as cópias (as 3 últimas):")
for c in capturas[-3:]:
    print(f"   https://web.archive.org/web/{c['timestamp']}/{c['original']}")
print(f"   HTML cru: https://web.archive.org/web/{ultima['timestamp']}id_/{ultima['original']}")

### `br/03_linha_do_tempo_caso.py` — Caso Vitrine Fantasma: a linha do tempo do golpe e o efeito Black Friday.

In [ ]:
%cd -q {PASTA_NB}
"""
Caso Vitrine Fantasma: a linha do tempo do golpe e o efeito Black Friday.

Carta 40 (rank A) — Jutsu da Linha do Tempo (aplicação no caso)

O que você aprende:
  • a juntar eventos de fontes diferentes (RDAP, Wayback, denúncias) numa única linha do tempo;
  • a ler o "length" das capturas: página que encolhe de repente = "em manutenção";
  • a calcular a Black Friday com o módulo calendar (o dia seguinte à 4ª quinta-feira de novembro);
  • a medir a concentração de denúncias perto dela.

100% offline: usa os dados fictícios de caso/dados/.

Como rodar (dentro da pasta dia-19-wayback-e-datas):
    python br/03_linha_do_tempo_caso.py
"""

import calendar
import csv
import json
from collections import Counter
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

# ── Parâmetros ────────────────────────────────────────────────────────────
JANELA_DIAS = 3  #@param {type:"integer"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"

try:
    from zoneinfo import ZoneInfo
    FUSO = ZoneInfo("America/Sao_Paulo")
except Exception:
    FUSO = timezone(timedelta(hours=-3))

DIAS_SEMANA = ["seg", "ter", "qua", "qui", "sex", "sáb", "dom"]


def black_friday(ano):
    """Black Friday = a sexta-feira logo depois da 4ª quinta-feira de novembro (o Thanksgiving dos EUA).

    calendar.monthcalendar devolve as semanas do mês como listas de 7 números (0 = dia de outro mês).
    """
    semanas = calendar.monthcalendar(ano, 11)
    quintas = [s[calendar.THURSDAY] for s in semanas if s[calendar.THURSDAY] != 0]
    return date(ano, 11, quintas[3]) + timedelta(days=1)


eventos = []   # cada evento: (data, ícone, descrição)

# ── 1. RDAP: nascimento e alteração do domínio ────────────────────────────
rdap = json.loads((PASTA_CASO / "rdap_megaoferta.json").read_text(encoding="utf-8"))
nomes = {"registration": "Domínio registrado", "last changed": "Registro do domínio alterado"}
for e in rdap["events"]:
    if e["eventAction"] in nomes:
        quando = datetime.fromisoformat(e["eventDate"].replace("Z", "+00:00")).astimezone(FUSO)
        eventos.append((quando.date(), "🌐", f"{nomes[e['eventAction']]} (RDAP, {quando:%H:%M})"))

# ── 2. Wayback: capturas e o que o "length" revela ────────────────────────
cdx = json.loads((PASTA_CASO / "wayback_cdx_megaoferta.json").read_text(encoding="utf-8"))
cabecalho, *linhas = cdx                        # "desempacota": 1º item no cabeçalho, o resto em linhas
capturas = [dict(zip(cabecalho, l)) for l in linhas]
tamanho_anterior = None
for c in capturas:
    quando = datetime.strptime(c["timestamp"], "%Y%m%d%H%M%S").replace(tzinfo=timezone.utc).astimezone(FUSO)
    tamanho = int(c["length"])
    nota = f"captura Wayback · HTTP {c['statuscode']} · {tamanho} bytes"
    if c["statuscode"] == "404":
        nota += " → 🚩 SITE FORA DO AR"
    elif tamanho_anterior and tamanho < tamanho_anterior * 0.3:
        # Encolheu mais de 70%: a vitrine com produtos virou uma página quase vazia.
        nota += f" → 🚩 encolheu {1 - tamanho / tamanho_anterior:.0%} (página de 'manutenção'?)"
    eventos.append((quando.date(), "📸", nota))
    tamanho_anterior = tamanho

# ── 3. Denúncias: primeira, última e picos ────────────────────────────────
with open(PASTA_CASO / "denuncias.csv", encoding="utf-8") as arquivo:
    datas = [date.fromisoformat(l["data"]) for l in csv.DictReader(arquivo, delimiter=";")]
por_dia = Counter(datas)
eventos.append((min(datas), "📨", "Primeira denúncia"))
eventos.append((max(datas), "📨", "Última denúncia"))
for dia, qtd in por_dia.items():
    if qtd >= 4:
        eventos.append((dia, "📈", f"Pico: {qtd} denúncias num só dia"))

# ── 4. Black Friday ───────────────────────────────────────────────────────
bf = black_friday(2025)
eventos.append((bf, "🛒", "BLACK FRIDAY 2025"))

# ── 5. Imprimir a linha do tempo ordenada ─────────────────────────────────
# sorted() em tuplas compara a data primeiro; em empate, o ícone (só para ficar estável).
print("🗓️ LINHA DO TEMPO — Operação Vitrine Fantasma\n")
criacao = min(d for d, i, _ in eventos if i == "🌐")
for dia, icone, texto in sorted(eventos):
    print(f"   {dia:%d/%m/%Y} {DIAS_SEMANA[dia.weekday()]}  D+{(dia - criacao).days:<3} {icone} {texto}")

# ── 6. O efeito Black Friday ──────────────────────────────────────────────
perto = sum(q for d, q in por_dia.items() if abs((d - bf).days) <= JANELA_DIAS)
dias_janela = 2 * JANELA_DIAS + 1
periodo = (max(datas) - min(datas)).days + 1
print(f"\n🛒 Black Friday 2025: {bf:%d/%m/%Y} (sexta após a 4ª quinta de novembro)")
print(f"   {perto} de {len(datas)} denúncias ({perto / len(datas):.0%}) caíram entre "
      f"{bf - timedelta(JANELA_DIAS):%d/%m} e {bf + timedelta(JANELA_DIAS):%d/%m}: "
      f"{dias_janela} dias de um período de {periodo}.")
media_geral = len(datas) / periodo
media_janela = perto / dias_janela
print(f"   Média: {media_janela:.1f} denúncias/dia na janela × {media_geral:.1f}/dia no período todo "
      f"({media_janela / media_geral:.1f}x).")
print("\n💡 Leitura: o golpe nasceu 10 dias antes da 1ª vítima, acelerou na semana da Black Friday,")
print("   virou 'manutenção' em 01/12 e saiu do ar em 05/12. Roteiro clássico de loja falsa sazonal.")

## ⚖️ Ética, LGPD e OPSEC

- **O arquivo também guarda dados pessoais.** Páginas antigas podem mostrar telefone, e-mail ou foto
  de alguém que depois os apagou de propósito. Use só o necessário para a finalidade e não
  republique; o direito de apagar não some só porque existe uma cópia.
- **Consultar a Wayback não alerta o alvo**: a requisição vai para o Internet Archive, não para o site
  do suspeito. Já abrir o site vivo deixa o seu IP nos logs dele.
- **Capture antes que suma.** Num golpe ativo, peça a captura (Save Page Now) e salve você também uma
  cópia com hash (Dia 14) e data/hora com fuso. A Wayback pode não ter a página que você precisa.
- **Captura é indício, não certidão.** Para uso em processo, registre a URL do memento, o timestamp,
  o hash do conteúdo baixado e como você o obteve. Algumas páginas são capturadas incompletas
  (sem imagens ou sem JavaScript).
- Respeite os limites da API: nada de laço sem pausa baixando milhares de mementos.

## 🎯 Quiz rápido

Escolha a resposta no formulário de cada célula e rode a célula.

In [ ]:
#@title 1. Na resposta JSON da API CDX da Wayback Machine, o que é a primeira linha?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) A captura mais antiga", "B) A captura mais recente", "C) O cabeçalho com os nomes das colunas", "D) Um resumo com o total de capturas"]
correta = "C) O cabeçalho com os nomes das colunas"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "A primeira linha traz urlkey, timestamp, original, mimetype, statuscode, digest e length; os dados vêm depois.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 2. Por que não confiar em strftime('%A') para escrever o dia da semana em português?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Porque ele sempre erra o dia", "B) Porque o resultado depende do locale do sistema, que no Colab é inglês", "C) Porque só funciona no Windows", "D) Porque %A mostra o ano"]
correta = "B) Porque o resultado depende do locale do sistema, que no Colab é inglês"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "Uma lista própria indexada por weekday() dá o mesmo resultado em qualquer computador.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 3. Como o script calcula a Black Friday de um ano?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Sempre 25 de novembro", "B) Última sexta-feira de novembro", "C) A sexta-feira depois da 4ª quinta-feira de novembro", "D) Primeira sexta de dezembro"]
correta = "C) A sexta-feira depois da 4ª quinta-feira de novembro"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "A Black Friday segue o dia de Ação de Graças dos EUA (4ª quinta de novembro). Em 2024 isso não foi a última sexta.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

# 🏆 Desafio S — Dia 19: Que vitrine a vítima viu?

**Carta 61 · Rank S**

## A missão

A defesa vai perguntar: "Como vocês sabem o que o site mostrava quando a vítima comprou?". A
resposta honesta é: pela captura da Wayback **mais próxima antes** da compra. A Iara quer isso
calculado para cada uma das 37 denúncias.

Escreva um script que:

1. Leia `caso/dados/wayback_cdx_megaoferta.json` e converta cada `timestamp` (UTC) para o horário de
   Brasília.
2. Para cada denúncia, encontre a captura **mais recente com data menor ou igual** à da compra,
   usando o módulo **`bisect`** (busca binária), não um laço que testa todas.
3. Calcule a **defasagem** em dias entre a captura e a compra.
4. Conte quantas denúncias "viram" cada versão do site.
5. Liste as maiores **lacunas** entre capturas consecutivas e quantas denúncias caíram dentro delas.
6. Grave `saidas/vitrine_vista_por_denuncia.csv` (`;`, `utf-8-sig`) com o link do memento de cada uma.

## Critérios de rank S

- [ ] Só biblioteca padrão, 100% offline
- [ ] `bisect` sobre uma lista **ordenada** de datas
- [ ] Datas convertidas de UTC para America/Sao_Paulo (com plano B)
- [ ] Nenhuma denúncia associada a uma captura **posterior** à compra

## Dica

`bisect.bisect_right(lista, x)` devolve a posição onde `x` entraria, **depois** de itens iguais a
ele. Então `bisect_right(datas, dia) - 1` é o índice da última data `<= dia`. Se der `-1`, não
existe captura antes da compra.

In [ ]:
%cd -q {PASTA_NB}
# ✍️ Escreva sua solução aqui.

In [ ]:
#@title 🔒 Gabarito (dê dois cliques aqui para ver o código) { display-mode: "form" }
%cd -q {PASTA_NB}
"""
Gabarito do Desafio S do Dia 19 — Que vitrine a vítima viu?

Para cada denúncia, descobre qual captura da Wayback Machine é a mais recente ANTES (ou no
mesmo dia) da compra, isto é, a melhor evidência de como estava o site que a vítima viu.
Usa bisect (busca binária) numa lista ordenada de datas. Também mede as maiores lacunas entre
capturas (períodos em que NÃO temos prova de como o site estava) e grava um CSV.
100% offline: usa caso/dados/.
"""

import bisect                         # busca binária em lista ordenada
import csv
import json
from collections import Counter
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

MAIORES_LACUNAS = 3  #@param {type:"integer"}

PASTA_DIA = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"
SAIDAS = PASTA_DIA / "saidas"

try:
    from zoneinfo import ZoneInfo
    FUSO = ZoneInfo("America/Sao_Paulo")
except Exception:
    FUSO = timezone(timedelta(hours=-3))

# ── Capturas, em ordem, com data local ────────────────────────────────────
cabecalho, *linhas = json.loads((PASTA_CASO / "wayback_cdx_megaoferta.json").read_text(encoding="utf-8"))
capturas = []
for linha in linhas:
    c = dict(zip(cabecalho, linha))
    c["quando"] = datetime.strptime(c["timestamp"], "%Y%m%d%H%M%S").replace(tzinfo=timezone.utc).astimezone(FUSO)
    c["url_arquivo"] = f"https://web.archive.org/web/{c['timestamp']}/{c['original']}"
    capturas.append(c)
capturas.sort(key=lambda c: c["quando"])
datas_capturas = [c["quando"].date() for c in capturas]   # lista ORDENADA: requisito do bisect

# ── Para cada denúncia, a captura mais recente até o dia da compra ────────
with open(PASTA_CASO / "denuncias.csv", encoding="utf-8") as arquivo:
    denuncias = sorted(csv.DictReader(arquivo, delimiter=";"), key=lambda d: d["data"])

resultado = []
for d in denuncias:
    dia = date.fromisoformat(d["data"])
    # bisect_right devolve a posição onde 'dia' entraria mantendo a ordem, DEPOIS de iguais.
    # Então a posição - 1 é a última captura com data <= dia.
    i = bisect.bisect_right(datas_capturas, dia) - 1
    if i < 0:
        resultado.append({**d, "captura": "", "dias_de_defasagem": "", "length": "", "url_arquivo": ""})
        continue
    c = capturas[i]
    resultado.append({"id": d["id"], "data": d["data"], "produto": d["produto"],
                      "captura": c["timestamp"], "dias_de_defasagem": (dia - datas_capturas[i]).days,
                      "length": c["length"], "url_arquivo": c["url_arquivo"]})

versoes = Counter(r["captura"] for r in resultado)
print("🖼️ Qual versão arquivada cada vítima provavelmente viu:\n")
for c in capturas:
    qtd = versoes.get(c["timestamp"], 0)
    print(f"   {c['quando']:%d/%m/%Y %H:%M}  {c['length']:>5} bytes  HTTP {c['statuscode']}  ← {qtd:>2} denúncias")

pior = max(resultado, key=lambda r: r["dias_de_defasagem"] if r["dias_de_defasagem"] != "" else -1)
print(f"\n⏳ Maior defasagem: {pior['id']} ({pior['data']}) comprou {pior['dias_de_defasagem']} dias "
      f"depois da captura mais próxima. Quanto maior a defasagem, mais fraca a prova.")

# ── Lacunas entre capturas consecutivas ───────────────────────────────────
lacunas = sorted(((b["quando"] - a["quando"], a, b) for a, b in zip(capturas, capturas[1:])),
                 key=lambda x: x[0], reverse=True)
print(f"\n🕳️ As {MAIORES_LACUNAS} maiores lacunas (sem nenhuma captura):")
for delta, a, b in lacunas[:MAIORES_LACUNAS]:
    dentro = sum(1 for d in denuncias if a["quando"].date() < date.fromisoformat(d["data"]) < b["quando"].date())
    print(f"   {a['quando']:%d/%m} → {b['quando']:%d/%m}: {delta.days} dias, {dentro} denúncias no escuro")

SAIDAS.mkdir(exist_ok=True)
with open(SAIDAS / "vitrine_vista_por_denuncia.csv", "w", newline="", encoding="utf-8-sig") as f:
    escritor = csv.DictWriter(f, fieldnames=list(resultado[0]), delimiter=";", extrasaction="ignore")
    escritor.writeheader()
    escritor.writerows(resultado)
print("\n💾 saidas/vitrine_vista_por_denuncia.csv gravado.")
print("💡 Próximo passo real: pedir a captura (Save Page Now) das páginas de um golpe ativo ANTES que ele suma.")

## 🔎 Inovação OSINT do dia

**Um índice só para vários arquivos: `cdx_toolkit`.** A biblioteca `cdx_toolkit`
(`pip install cdx_toolkit`) consulta com a mesma interface o CDX da **Wayback Machine** e o do
**Common Crawl**, um rastreamento público da web feito várias vezes por ano. Quando a Wayback não
tem a página, o Common Crawl às vezes tem. E não depender de uma única fonte é lição aprendida: em
outubro de 2024 o Internet Archive sofreu ataques de negação de serviço e um vazamento de dados de
cerca de 31 milhões de contas, e ficou fora do ar por dias. Para OSINT, a regra fica: consulte mais de um arquivo e **guarde a sua cópia**.

## 📚 Para ir além

- Documentação da API CDX (servidor pywb/Wayback): <https://github.com/internetarchive/wayback/tree/master/wayback-cdx-server>
- Pacote `wayback` (EDGI): <https://wayback.readthedocs.io/>
- `datetime` e `zoneinfo`: <https://docs.python.org/pt-br/3/library/datetime.html> · <https://docs.python.org/pt-br/3/library/zoneinfo.html>
- `calendar`: <https://docs.python.org/pt-br/3/library/calendar.html>
- cdx_toolkit: <https://github.com/commoncrawl/cdx_toolkit>
- Save Page Now (capturar uma página agora): <https://web.archive.org/save>

## ✅ Checklist

- [ ] Rodei `date_time.py` e `download_mementos.py`
- [ ] Converti um timestamp CDX e um timestamp Unix em data de Brasília
- [ ] Vi o deslocamento -02:00 de dezembro de 2018
- [ ] Consultei a API CDX de uma página `.gov.br`
- [ ] Montei a linha do tempo do caso e achei a Black Friday
- [ ] Fiz o desafio S
- [ ] Anotei no meu [`DIARIO.md`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/DIARIO.md)

---

🎉 Terminou? [Registre sua entrega do Dia 19](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/issues/new?template=entreguei-o-dia.yml) · Baseado em [cipher387/python-for-OSINT-21-days](https://github.com/cipher387/python-for-OSINT-21-days) (MIT) · Adaptação pt-BR: [Ridd1kulusC0d3r/python-para-osint-21-dias](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias)